<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 1</b>

## Unit 5: RAG Evaluation & Advanced RAG

**CV Raman Global University, Bhubaneswar**  
*AI Center of Excellence*

</div>

---

### What You'll Learn

1. **Evaluate RAG retrieval** — measure ranking quality (MRR) and keyword coverage
2. **Evaluate RAG answers** — use an LLM-as-Judge to score accuracy, completeness, and relevance
3. **Diagnose RAG failures** — identify when embedding similarity ≠ relevance
4. **Query rewriting** — transform vague user questions into effective search queries
5. **Reranking** — reorder retrieved documents using LLM-based relevance scoring

**Duration:** ~1.5 hours

---

In [ ]:
# -- Setup --
!pip install -q openai

import json
from getpass import getpass
from openai import OpenAI

client = OpenAI(api_key=getpass("Enter your OpenAI API Key: "))
MODEL = "gpt-4o-mini"

---

## Quick RAG Recap

In Unit 4 we built a basic RAG pipeline:

```
[User Query] → [Embed] → [Vector Search] → [Top-k Docs] → [LLM + Context] → [Answer]
```

This works well for **specific, well-phrased queries** — but how do we know if it *actually* works? And what happens when it doesn't?

This notebook covers two things:
1. **How to measure** whether your RAG system is working (evaluation)
2. **How to fix it** when it isn't (advanced techniques)

---

## Why Evaluate RAG?

RAG can fail in different ways, and each failure requires a different fix:

| Failure Mode | What Went Wrong | Example | How You'd Know |
|-------------|----------------|---------|----------------|
| **Retrieval failure** | Wrong documents retrieved | Query about leave policy returns product docs | Low keyword coverage, low MRR |
| **Generation failure** | Right docs, wrong answer | Docs contain correct info but LLM hallucinates | Low accuracy score |
| **End-to-end failure** | Answer doesn't help user | Technically correct but misses the point | Low relevance score |

> **You can't improve what you can't measure.** Evaluation tells you *where* the pipeline is breaking so you can target your fixes.

### Two Sides of RAG Evaluation

```
RAG Evaluation
├── Retrieval Quality          ← Did we find the right documents?
│   ├── MRR (ranking quality)
│   └── Keyword Coverage
│
└── Answer Quality             ← Did we generate a good answer?
    ├── Accuracy
    ├── Completeness
    └── Relevance
```

---

## Retrieval Metrics — MRR & Keyword Coverage

### MRR — Mean Reciprocal Rank

MRR measures **how high** the first relevant document appears in the search results.

```
Rank of first relevant doc    →    Reciprocal Rank
─────────────────────────────────────────────────
        1st position           →    1/1 = 1.00
        2nd position           →    1/2 = 0.50
        3rd position           →    1/3 = 0.33
        Not found              →    0.00
```

**Worked example:** Search for *"Who founded TechSolutions?"* → 3 results:
1. "TechSolutions India was founded in 2018 by Priya Sharma..." ← **Relevant!** → RR = **1/1 = 1.0**
2. "Priya Sharma is the CEO and co-founder..."
3. "Major clients include HDFC Bank..."

If the relevant doc appeared at rank 2 instead, RR = **1/2 = 0.5**. Average across all test queries → **MRR**.

### Keyword Coverage

The simplest retrieval metric — did the retrieved documents contain the keywords we expected?

| Test Question | Expected Keywords | Found | Coverage |
|---------------|-------------------|-------|----------|
| "Who founded TechSolutions?" | Priya, Sharma, Rahul, Verma, 2018 | All 5 | **100%** |
| "What is the leave policy?" | 24, paid, 10, sick, leaves | 3 of 5 | **60%** |

**Formula:** `keywords_found / total_keywords × 100`

In [ ]:
# Demo: Retrieval metrics on simulated search results
# In a real RAG system, retrieved_docs would come from vector search (like we built in Unit 4).
# Here we hardcode them so we can focus purely on the metric calculation.

test_cases = [
    {
        "query": "Who founded TechSolutions India?",
        # Simulated: vector search returned these 3 docs for this query
        "retrieved_docs": [
            "TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma in Bhubaneswar.",
            "Priya Sharma is the CEO and co-founder. Education: IIT Delhi, Stanford MBA.",
            "Major clients: HDFC Bank, Tata Motors, Reliance Industries.",
        ],
        # We define expected_keywords as our ground truth for evaluation.
        # These are the keywords a correct retrieval result SHOULD contain --
        # we use them to measure how well the retrieval step performed.
        "expected_keywords": ["Priya", "Sharma", "Rahul", "Verma", "2018"],
    },
    {
        "query": "How much does CloudAssist Pro cost?",
        "retrieved_docs": [
            "CloudAssist Pro is the flagship cloud platform. Price: Rs.50,000/month.",
            "SmartHR is an AI-powered HR system. Price: Rs.25,000/month.",
            "TechSolutions has offices in Bhubaneswar, Bangalore, and Hyderabad.",
        ],
        "expected_keywords": ["50,000", "month", "CloudAssist"],
    },
    {
        "query": "What is the leave policy?",
        # Simulated: retrieval ranked the leave doc at position 3 (a common failure!)
        "retrieved_docs": [
            "Work hours: 9 AM to 6 PM. Hybrid model: 3 days office, 2 days remote.",
            "Health insurance: Rs.5 lakh coverage for employee and family.",
            "Leave policy: 24 paid leaves + 10 sick leaves per year.",
        ],
        "expected_keywords": ["24", "paid", "10", "sick", "leaves"],
    },
]

for case in test_cases:
    docs = case["retrieved_docs"]
    keywords = case["expected_keywords"]

    # MRR: find rank of first doc containing any expected keyword
    mrr = 0.0
    for rank, doc in enumerate(docs, 1):
        if any(kw.lower() in doc.lower() for kw in keywords):
            mrr = 1.0 / rank
            break

    # Keyword coverage: what fraction of expected keywords appear in all docs combined
    combined = " ".join(docs).lower()
    found = sum(1 for kw in keywords if kw.lower() in combined)
    coverage = found / len(keywords) * 100

    print(f"Q: {case['query']}")
    print(f"  MRR: {mrr:.2f}  |  Keyword Coverage: {found}/{len(keywords)} = {coverage:.0f}%\n")

---

## LLM-as-Judge (Answer Quality)

Instead of manually checking every answer, we use an LLM to rate the generated answer against a reference answer:

| Dimension | What It Measures | Scale |
|-----------|-----------------|-------|
| **Accuracy** | Is the information factually correct? | 1–5 |
| **Completeness** | Does it cover all key points from the reference? | 1–5 |
| **Relevance** | Does it actually answer the question asked? | 1–5 |

This approach is widely used in practice — it's fast, scalable, and [correlates well with human judgment](https://arxiv.org/abs/2306.05685). Be aware of potential biases: LLM judges can prefer longer or more verbose answers.

For production RAG evaluation at scale, see the [RAGAS framework](https://docs.ragas.io/).

In [ ]:
# Demo: LLM-as-Judge scores a simulated RAG answer
# In practice, both the reference and generated answers come from your RAG pipeline.
# Here we hardcode them to show how the judge prompt works.

question = "Who founded TechSolutions India and when?"
reference_answer = "Priya Sharma and Rahul Verma founded TechSolutions India in 2018 in Bhubaneswar, Odisha."
rag_generated_answer = "TechSolutions India was founded by Priya Sharma in 2018."  # intentionally incomplete

judge_prompt = f"""Evaluate this RAG-generated answer against the reference.

Question: {question}
Reference Answer: {reference_answer}
Generated Answer: {rag_generated_answer}

Rate on a scale of 1-5:
- Accuracy: Is the information correct?
- Completeness: Does it cover all key points?
- Relevance: Does it answer the question?

Respond in this exact format:
ACCURACY: [1-5]
COMPLETENESS: [1-5]
RELEVANCE: [1-5]
FEEDBACK: [One sentence explanation]"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": judge_prompt}],
    temperature=0,
)
result = response.choices[0].message.content

print(f"Question:  {question}")
print(f"Reference: {reference_answer}")
print(f"Generated: {rag_generated_answer}")
print(f"\nJudge scores:\n{result}")

---

## The Problem with Basic RAG

Basic RAG assumes **embedding similarity = relevance**. In practice, this breaks down in two ways:

### Problem 1: Embeddings Miss Nuance

A document about *"PhD research at MIT"* may not rank highest for *"Who has advanced degrees?"* — embedding spaces don't perfectly capture semantic equivalence.

### Problem 2: Users Don't Write Search Queries

| What users actually ask | What would retrieve better results |
|------------------------|-----------------------------------|
| "What do I get if I join?" | "employee benefits compensation packages" |
| "Tell me about the big boss" | "CEO founder TechSolutions leadership" |
| "How much money can I make extra?" | "performance bonus salary compensation" |

### A Simulated Failure

Imagine searching a company knowledge base with the vague query *"What do I get if I join?"*

| Rank | Document Retrieved | Relevant? |
|------|-------------------|-----------|
| 1 | "TechSolutions has offices in Bhubaneswar, Bangalore, and Hyderabad..." | No |
| 2 | "Completed 200+ projects across banking, automotive, retail..." | No |
| 3 | "Health insurance: ₹5 lakh coverage for employee and family..." | Yes! |

The relevant doc landed at rank 3 (MRR = 0.33), and the documents about learning budget, performance bonus, and gym membership were missed entirely. The answer will be incomplete.

---

## Query Rewriting

**Query rewriting** transforms a user's natural-language question into a more effective search query *before* retrieval.

| User says | Rewritten query |
|-----------|------------------|
| "What do I get if I join?" | "employee benefits compensation TechSolutions" |
| "Tell me about the big boss" | "CEO founder TechSolutions India leadership" |
| (after asking about Priya) "Where did she study?" | "Priya Sharma education university degree" |

The third example shows **coreference resolution** — the rewriter resolves "she" to "Priya Sharma" using conversation history.

Further reading: [Query Rewriting for Retrieval-Augmented LLMs (Ma et al., 2023)](https://arxiv.org/abs/2305.14283)

In [ ]:
# Demo: Query rewriting with LLM

rewrite_prompt = """You are helping improve search queries for a company knowledge base.
Rewrite the user's question as a clear, specific search query.
If conversation history is provided, resolve any pronouns (he/she/it/they).
Return ONLY the rewritten query, nothing else."""

# Rewrite vague queries
vague_queries = [
    "What do I get if I join?",
    "Tell me about the big boss",
    "How much money can I make extra?",
]

print("Query Rewriting:\n")
for q in vague_queries:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": rewrite_prompt},
            {"role": "user", "content": q},
        ],
        temperature=0,
    )
    print(f"  Original:  {q}")
    print(f"  Rewritten: {r.choices[0].message.content}\n")

# Coreference resolution with conversation history
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": rewrite_prompt},
        {"role": "user", "content": "Tell me about Priya"},
        {"role": "assistant", "content": "Priya Sharma is the CEO of TechSolutions India."},
        {"role": "user", "content": "Where did she study?"},
    ],
    temperature=0,
)
print("Coreference resolution (user previously asked about Priya):")
print(f"  Follow-up: Where did she study?")
print(f"  Rewritten: {r.choices[0].message.content}")

---

## Reranking

**Reranking** is a second-pass relevance scoring step applied after retrieval.

**Key insight:** Embedding models encode documents *independently* of any query (bi-encoder). A reranker evaluates each **query–document pair** jointly (cross-encoder), producing a more accurate relevance judgment.

```
Bi-encoder (retrieval):    Query → [Embed]    Doc → [Embed]    → cosine sim  (fast, approximate)
Cross-encoder (reranking): [Query + Doc] → single model →      relevance     (slow, precise)
```

### Why Independent Encoding Fails

Bi-encoders embed the query and document **separately** — they never see each other's words. This means relevance depends entirely on how close the two vectors land in embedding space. When the query and document use different vocabulary to express the same concept, cosine similarity breaks down:

- **Query:** *"Who has advanced degrees?"* → embedding captures "advanced", "degrees"
- **Document:** *"She holds a PhD from IISc Bangalore"* → embedding captures "PhD", "IISc", "Bangalore"
- **Cosine similarity is low** because the *words* are different, even though the *meaning* matches perfectly

A cross-encoder reads the query and document **together** as a single input. It sees both "advanced degrees" and "PhD" in the same context and understands that a PhD **is** an advanced degree. This joint reasoning is what makes reranking more accurate.

### The Retrieve-Then-Rerank Cascade

If cross-encoders are more accurate, why not use them for everything?

Because cross-encoders score **one (query, doc) pair at a time**. If your knowledge base has 10,000 documents, you'd need 10,000 forward passes through the model — far too slow for real-time search.

The solution is a two-stage cascade:

```
10,000 docs ──→ Bi-encoder (fast) ──→ Top 5 ──→ Cross-encoder (precise) ──→ Reranked Top 5
                  ~10ms                              ~200ms
```

1. **Stage 1 — Retrieve:** Use the fast bi-encoder to scan all 10,000 documents and return the top-k (e.g., 5)
2. **Stage 2 — Rerank:** Use the slow-but-accurate cross-encoder to reorder only those 5 documents

This gives us the **speed of embeddings** + the **accuracy of cross-encoders**.

### When Reranking Helps — and When It Doesn't

- **Helps:** The relevant documents *were* retrieved but are **ranked poorly**. This is exactly the scenario in the Cell 12 demo — the PhD docs are in the top 5, just not at positions 1 and 2. Reranking pushes them to the top.

- **Doesn't help:** The relevant documents **weren't retrieved at all**. If the bi-encoder missed the right documents entirely, reranking 5 wrong documents still gives you 5 wrong documents.

In evaluation terms: reranking improves **precision** (the ordering quality of retrieved results), not **recall** (whether the relevant document was found in the first place). If recall is the problem, you need better embeddings, query rewriting, or a larger top-k.

> **Reranking vs Query Rewriting:** Query rewriting fixes *what you're searching for* (the input to retrieval). Reranking fixes *how the results are ordered* (the output of retrieval). They solve different problems and work well together.

### Reranking Approaches

| Approach | Examples | Pros | Cons |
|----------|----------|------|------|
| **Cross-encoder models** | Cohere Rerank, `ms-marco-MiniLM` | Fast, cheap, purpose-built | Requires separate model |
| **LLM-based reranking** | GPT-4o, Claude as judge | No extra model, high quality | Slower, more expensive |

We use LLM-based reranking here for simplicity. In production, dedicated cross-encoders are preferred.

Further reading: [Passage Re-ranking with BERT (Nogueira & Cho, 2019)](https://arxiv.org/abs/1901.04085)

In [ ]:
# Demo: LLM-based reranking of simulated retrieval results
# These 5 docs simulate what vector search returned for "Who has a PhD?"
# Notice the relevant docs (2, 4) are buried -- reranking fixes this.

query = "Who has a PhD?"
retrieved_docs = [
    "TechSolutions has offices in Bhubaneswar, Bangalore, and Hyderabad.",
    "Ananya Patel is VP of Engineering. She holds a PhD from IISc Bangalore in Computer Science.",
    "CloudAssist Pro is the flagship cloud platform. Price: Rs.50,000/month.",
    "Vikram Reddy is Head of AI Research. He completed his PhD from MIT.",
    "Priya Sharma is the CEO. Education: IIT Delhi (B.Tech), Stanford (MBA).",
]

docs_text = "\n".join(f"[Doc {i+1}]: {doc}" for i, doc in enumerate(retrieved_docs))
rerank_prompt = f"""Given a question and retrieved documents, reorder them from most to least relevant.

Question: {query}

Documents:
{docs_text}

Return a JSON array of document numbers in order of relevance. Example: [3, 1, 5, 2, 4]
Only return the JSON array, nothing else."""

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": rerank_prompt}],
    temperature=0,
)
order = json.loads(r.choices[0].message.content.strip())

print(f"Query: {query}\n")
print("Before reranking:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  {i}. {doc}")

print(f"\nReranked order: {order}\n")
print("After reranking:")
for rank, idx in enumerate(order, 1):
    print(f"  {rank}. {retrieved_docs[idx - 1]}")

---

## Putting It Together: Advanced RAG Pipeline

The full advanced pipeline chains all components:

```
User Query → Rewrite Query → Retrieve 5 docs → Rerank → Top 3 → LLM → Answer
```

We retrieve 5 documents but only pass the top 3 (after reranking) to the LLM. This balances **recall** (casting a wide net) with **precision** (keeping context focused).

### "Lost in the Middle"

Research by [Liu et al. (2023)](https://arxiv.org/abs/2307.03172) showed that LLMs perform best when key information is at the **beginning or end** of the context, not buried in the middle. This is why retrieve-broad-then-rerank (5 → 3) works better than just sending all 5 documents to the LLM.

### Basic vs Advanced RAG — Side-by-Side

| Query | Basic RAG | Advanced RAG |
|-------|-----------|---------------|
| "What do I get if I join?" | Retrieves office locations, project history. Misses benefits. | Rewrites to "employee benefits compensation". Retrieves health insurance, learning budget, bonus docs. |
| "Who went to the best schools?" | Retrieves company overview. Misses leadership bios. | Rewrites to "education qualifications university". Retrieves Priya (IIT+Stanford), Rahul (BITS+IIM), Vikram (MIT). |
| "What's the deal with working from home?" | Retrieves product info. Misses work policy. | Rewrites to "remote work hybrid policy". Retrieves hybrid model (3 office, 2 remote). |

---

## When to Use What

| Technique | Best For | Latency | API Cost | Complexity |
|-----------|----------|---------|----------|------------|
| **Basic RAG** | Specific, well-phrased queries | Low | 1 LLM call | Simple |
| **+ Reranking** | Noisy retrieval, precision-critical apps | +100–500ms | +1 LLM call | Moderate |
| **+ Query Rewriting** | Conversational UI, vague user queries | +100–300ms | +1 LLM call | Moderate |
| **Both** | Production systems, general-purpose chatbots | +200–800ms | +2 LLM calls | Higher |

**Production tip:** For reranking at scale, use dedicated cross-encoder models (e.g., [Cohere Rerank](https://docs.cohere.com/docs/rerank), `cross-encoder/ms-marco-MiniLM-L-6-v2`) instead of LLM-based reranking — they're faster, cheaper, and purpose-built.

---

## Key Takeaways

1. **You can't improve what you can't measure** — evaluate both retrieval (MRR, keyword coverage) and answer quality (LLM-as-Judge) separately to pinpoint failures.

2. **Embedding similarity ≠ relevance** — reranking with a cross-encoder or LLM evaluates query–document pairs jointly, catching what embeddings miss.

3. **Users don't write search queries** — query rewriting bridges the gap between natural conversation and effective retrieval.

4. **Retrieve broad, use narrow** — retrieve more documents than you need (5), rerank, then pass only the best (3) to the LLM.

5. **Each technique adds latency and cost** — apply them where they matter: reranking for precision, query rewriting for conversational interfaces.

---

### Resources

**Interactive tools:**
- [Embedding Projector](https://projector.tensorflow.org/) — visualize high-dimensional embeddings
- [RAGAS Framework](https://docs.ragas.io/) — production-grade RAG evaluation
- [Building and Evaluating Advanced RAG (DeepLearning.AI)](https://www.deeplearning.ai/short-courses/building-evaluating-advanced-rag/) — free short course

**Papers:**
- [Judging LLM-as-a-Judge with MT-Bench (Zheng et al., 2023)](https://arxiv.org/abs/2306.05685)
- [Retrieval-Augmented Generation for LLMs: A Survey (Gao et al., 2024)](https://arxiv.org/abs/2312.10997)
- [Passage Re-ranking with BERT (Nogueira & Cho, 2019)](https://arxiv.org/abs/1901.04085)
- [Query Rewriting for Retrieval-Augmented LLMs (Ma et al., 2023)](https://arxiv.org/abs/2305.14283)
- [Lost in the Middle (Liu et al., 2023)](https://arxiv.org/abs/2307.03172)

**Guides:**
- [Advanced RAG Techniques (Pinecone)](https://www.pinecone.io/learn/advanced-rag-techniques/)
- [Cohere Rerank Documentation](https://docs.cohere.com/docs/rerank)
- [Query Transformations (LangChain Blog)](https://blog.langchain.dev/query-transformations/)

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 1
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---